<h1><center>Diabetes - Classification Prediction </center></h1>

# Abstract
Diabetes is a chronic disease that occurs when the pancreas is no longer able to make insulin, or when the body cannot make good use of the insulin it produces. Learning how to use Machine Learning can help us predict Diabetes.

Keywords: python, seaborn, numpy, plotly, pandas, matplotlib, scikit-learning, decision tree, random forest, logistic regression, machine learning, supervised learning

# About this Project
This project seeks to better understand the various factors that affect diabetes and their relationship with each other.  The project will also use various machine learning techniques to predict various life expectancies by country and other indicators and provide the metrics needed to evaluate the predictions.

<li>The objective of this project is to classify whether someone has diabetes or not.</li>
<li>Dataset consists of several Medical Variables(Independent) and one Outcome Variable(Dependent)</li>
<li>The independent variables in this data set are :-'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin','BMI', 'DiabetesPedigreeFunction', 'Age'</li>
<li>The outcome variable value is either 1 or 0 indicating whether a person has diabetes(1) or not(0).</li>


# The Data
This dataset is a part of a large dataset held at the National Institute of Diabetes and Digestive and Kidney Diseases and was obtained through https://data.world/

The target variable is specified as "outcome", where 1 indicates a positive result of the diabetes test and 0 indicates a negative result.

A machine learning model is requested to be developed that can predict whether individuals have diabetes or not when their features are given. Before developing the model, performing the necessary data analysis and feature engineering steps are expected.

## There are 9 variables in the dataset:

<li>Pregnancies: Number of pregnancies.</li>
<li>Glucose: 2-hour plasma glucose concentration during an oral glucose tolerance test.</li>
<li>Blood Pressure: Blood Pressure (Diastolic) (mm Hg).</li>
<li>SkinThickness : Thickness of skin</li>
<li>Insulin: 2-Hour Serum Insulin (mu U/ml)</li>
<li>DiabetesPedigreeFunction: Function (2-hour plasma glucose concentration during an oral glucose tolerance test).</li>
<li>BMI: Body Mass Index (weight in kg/(height in meters)^2)</li>
<li>Age: Persons age(in years)</li>
<li>Outcome: Having diabetes status (1: diabetes, 0: no diabetes)</li>

# Importing Python libraries

In [ ]:
# Import libraries
import os
import csv

# Import packages for EDA and Data Visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Visualizatioin
import plotly.graph_objects as go
from pandas.plotting import scatter_matrix
import dataframe_image as dfi
from IPython.display import Image
import hvplot.pandas

#Statistical Functions
from IPython.display import display, HTML
from scipy.stats import stats
from scipy.stats.mstats import winsorize

# Decision Tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier 
from sklearn.tree import export_graphviz
from io import StringIO
import pydotplus
import six
import sys
sys.modules['sklearn.externals.six'] = six
from IPython.display import Image 

# Model Traning and Feature Exploration / Engineering 
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split, cross_val_score
from scipy.stats import mstats
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV
dt = DecisionTreeClassifier(random_state=42)

# Logistic Models & Metrics
import numpy as np
from sklearn import metrics
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix, accuracy_score, \
    classification_report, roc_auc_score, roc_curve
from sklearn.preprocessing import PolynomialFeatures
#from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression, LogisticRegression
from ydata_profiling import ProfileReport

# K Nearest Neighbour
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RepeatedStratifiedKFold

# Naive Bayes
from sklearn.naive_bayes import GaussianNB

#Support Vector Machine
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Decision Tree
from sklearn.tree import export_graphviz
from sklearn.externals.six import StringIO  
import pydotplus

#Gradient Boosting Classifier
from sklearn.ensemble import GradientBoostingClassifier

SEED = 42  # random seed for modeling

# Global Variables & Settings

In [ ]:
# Pandas settings
pd.set_option('display.width', 480)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option("display.max_columns", 20)

feature_cols = ['Pregnancies', 'Glucose','BloodPressure','SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


# Check for directory called charts, create it if it doesn't exist
directory = "charts"

if not os.path.exists(directory):
    os.makedirs(directory)
    print(f"Directory '{directory}' created.")
else:
    print(f"Directory '{directory}' already exists.")

# Loading Dataset

In [ ]:
# Creating pandas dataframe
df = pd.read_csv('diabetes.csv')
num_cols = [col for col in df.columns if df[col].dtypes != "O"]

# Function Definitions

In [ ]:
# Prints charts to image files
def view_df_to_image(text_body, file_name):
    df_styled = text_body.style.background_gradient()  # Style the text
    dfi.export(df_styled, 'charts/' + file_name + '.png')  # Create image of text

In [ ]:
# Provides breakdown of nulls
def nulls_breakdown(df=df):
    df_cols = list(df.columns)
    cols_total_count = len(list(df.columns))
    cols_count = 0
    for loc, col in enumerate(df_cols):
        null_count = df[col].isnull().sum()
        total_count = df[col].isnull().count()
        percent_null = round(null_count/total_count*100, 2)
        if null_count > 0:
            cols_count += 1
            print('[iloc = {}] {} has {} null values: {}% null'.format(loc, col, null_count, percent_null))
    cols_percent_null = round(cols_count/cols_total_count*100, 2)
    print('Out of {} total columns, {} contain null values; {}% columns contain null values.'.format(cols_total_count, cols_count, cols_percent_null))

In [ ]:
# Create visuals for outliers
def outliers_visual(data):
    plt.figure(figsize=(15, 40))
    i = 0
    for col in cont_vars:
        i += 1
        plt.subplot(9, 4, i)
        plt.boxplot(data[col])
        plt.title('{} boxplot'.format(col))
        i += 1
        plt.subplot(9, 4, i)
        plt.hist(df_selected[col])
        plt.title('{} histogram'.format(col))
    plt.show()

# Exploratory Data Analysis

In exploratory data analysis, the type of the variables (e.g. integer or decimal) is first printed, the sum of the missing values (either null or na) is calculated, and it is found that no such value exists.
In addition, some statistical measures are calculated to understand their variation. Some of these statistical measures are the mean of each column, the median, the standard deviation, the percentages, etc.

### Understanding the Dataset

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.head(5)

In [ ]:
df.tail(5)

In [ ]:
# Write charts to image files    
# Write dataframe info to images
dfi.export(pd.DataFrame(df.dtypes).style.background_gradient(), 'charts/Orig_Column_Headings.png')

# Write head() to image
text_body = df.head(5)
view_df_to_image(text_body, 'Head(5)')
df_styled = text_body.style.background_gradient()  # Style the text
dfi.export(df_styled, 'charts/' + 'Head(5)' + '.png')  # Create image of text

# Write tail() to images
text_body = df.tail(5)
view_df_to_image(text_body, 'Tail(5)')
df_styled = text_body.style.background_gradient()  # Style the text
dfi.export(df_styled, 'charts/' + 'Tail(5)' + '.png')  # Create image of text

In [ ]:
# Check for duplicate values
df.duplicated().any()

No duplicates records found, we have a clean dataframe.

In [ ]:
# Describe view stats on dataframe
pd.set_option("display.float", "{:.2f}".format)
df.describe()

<b>Conclusion:</b> We observe that min value of some columns is 0 which cannot be possible medically.Hence in the data cleaning process we'll have to replace them with median/mean value depending on the distribution. Also in the max column we can see insulin levels as high as 846 with a mean of only 79.80. We have to treat outliers. Skinthickness also shows a large variance between the mean of 20.53 and the max of 99.

The breakdown for our dependant variable (Outcome is shown below.  1 represent diabetes being present while 0 indicates no diabetes.

### Binary Classification Imbalance
Next, let's take a look at the proportion of classes. We can view the outcome distrubution and counts with bar and pie charts.  0 equals no diabetes while 1 equals diabetes present. 

In [ ]:
df.Outcome.value_counts()

In [ ]:
f,ax=plt.subplots(1,2,figsize=(7,3))
df['Outcome'].value_counts().plot.pie(explode=[0,0.1],autopct='%1.1f%%',ax=ax[0],shadow=True)
ax[0].set_title('Target Feature')
#ax[0].set_ylabel('Outcome')
sns.countplot(x='Outcome',data=df)
ax[1].set_title('Target Feature')
plt.show()

<b>Conclusion:</b> We observe that number of people who do not have diabetes (500) is far more than people who do (268) which indicates that our data is imbalanced.

### Univariant Analysis 

Now we will examine the distrubtion of all the attributes in the dataset by Outcome - the orange data points represent "have diabetes" and blue represents "do not have diabetes".

In [ ]:
sns.pairplot(df, hue='Outcome')
plt.show()

<b>Observations:</b>:

<li>There are less younger women that have diabetes, compared to women in their 30s-50s.</li>
<li>Women that have diabetes have higher BMI.</li>
<li>Diabetes pedigree function is lower for women that have diabetes; Same thing with insulin & glucose.</li>
<li>Pregnacies are also lower among women that have diabetes.</li>
<li>There are outliers present in almost all of the attributes.</li>

<b>Note:</b> Women are those known to have had pregnacies

### Duplicate Records

In [ ]:
# Check for duplicate rows
duplicate_rows = df[df.duplicated()]
num_duplicates = len(duplicate_rows)

# Print the number of duplicates found or zero if none are found
print("Number of duplicate rows found:", num_duplicates if num_duplicates > 0 else 0)

# Dropping duplicate values - checking if there are any duplicate rows and dropping if any
df=df.drop_duplicates()

### Null Summary
Missing values can exist when we have nulls, but there can also be erroneous missing values when the values appear to be inexplicit. Inexplicit nulls may require a more discretion when deciding how to deal with them as they may still represent actual data. "isnull" reveals that we have no null values.

In [ ]:
nulls_breakdown()

Checking for 0 values for our variables, Age & DiabetesPedigreeFunction do not have have minimum 0 value so no need to replace, also number of pregnancies as 0 is possible as observed in df.describe

In [ ]:
print(df[df['BloodPressure']==0].shape[0])
print(df[df['Glucose']==0].shape[0])
print(df[df['SkinThickness']==0].shape[0])
print(df[df['Insulin']==0].shape[0])
print(df[df['BMI']==0].shape[0])

<b>Conclusion:</b> Some of the columns have a skewed distribution, so the mean is more affected by outliers than the median. Glucose and Blood Pressure have normal distributions hence we replace 0 values in those columns by mean value. SkinThickness, Insulin,BMI have skewed distributions hence median is a better choice as it is less affected by outliers.

## Missing Value Imputation 

### Before Imputation

With imputation we are looking for zero values in 4 specific columns - bloodPressure, BMI, glucose, and skinthickness as none of them should contains zero values. Columns like insulin and pregnancies may legitimately contain zero values. 

In [ ]:
# Define columns with missing values
missing_colsDF = df[['BloodPressure','BMI', 'Glucose', 'SkinThickness']]
    
# Show the pairplot
sns.pairplot(missing_colsDF)
plt.show()


Looking at the charts above we can see all the zero values 

To address the zero values we'll imputate them with median values.

In [ ]:
#replacing 0 values with median of that column
df['Glucose']=df['Glucose'].replace(0,df['Glucose'].mean())#normal distribution
df['BloodPressure']=df['BloodPressure'].replace(0,df['BloodPressure'].mean())#normal distribution
df['SkinThickness']=df['SkinThickness'].replace(0,df['SkinThickness'].median())#skewed distribution
df['Insulin']=df['Insulin'].replace(0,df['Insulin'].median())#skewed distribution
df['BMI']=df['BMI'].replace(0,df['BMI'].median())#skewed distribution

### After Imputation

In [ ]:
# Define columns with missing values
missing_colsDF = df[['BloodPressure','BMI', 'Glucose', 'SkinThickness']]
    
# Show the pairplot
sns.pairplot(missing_colsDF)
plt.show()

Comparing the before and after imputation charts we can see the disappearance of the zero values givng us more reliable data.

## Correlation
Now is a good time to explore the correlation between the dependent variable (Outcome) and the primary independent variables (Glucose, BMI, BloodPressure, SkinThickness, and Insulin).

First we'll use a correlation matrix to observe linear relationships between variables.

In [ ]:
# Create correlation matrix
df.corr()

A heat map is a two-dimensional representation of the same information with the help of colors. Heat maps can help the user visualize simple or complex information.

In [ ]:
# Create the correlation matrix
corr = df.corr()

# Generate a mask for the upper triangle 
mask = np.triu(np.ones_like(corr, dtype=bool))

# Add the mask to the heatmap
#plt.figure(figsize = (8,4))
#sns.heatmap(corr, mask=mask, center=0, linewidths=1, annot=True, fmt=".2f", cmap="Blues")
#plt.show()
sns.set(rc = {'figure.figsize':(8,4)})
sns.heatmap(df.corr(), annot = True, fmt='.2g',cmap= 'coolwarm')

<b>Conclusion:</b> >Observe the last row 'Outcome' and note its correlation scores with different features. We can observe that Glucose, Glucose, BMI, Age, and Pregnancies are the most correlated with Outcome. BloodPressure, Insulin, DiabetesPedigreeFunction, and SkinThickness are the least correlated so we can drop them.

Before we dro the least correlated variables we can use blox plots to view correlation in a different way.

In [ ]:
sns.boxplot(x='Outcome',y='Glucose',data=df)

Above we can see the high correlation between Glucose and diabetes.  Those with diabetes show mucher levels of Glucose.

In [ ]:
sns.boxplot(x='Outcome', y='BMI', data=df)

BMI and diabetes show a slight correlation with a slightly BMI for those with diabetes.

In [ ]:
sns.boxplot(x='Outcome',y='BloodPressure',data=df)

No correlation is shown between BloodPressure and diabetes.

In [ ]:
sns.boxplot(x='Outcome',y='SkinThickness',data=df)

No correlation is shown between SkinThickness and diabetes.

In [ ]:
sns.boxplot(x='Outcome',y='Insulin',data=df)

No correlation is shown between Insulin and diabetes.

<b>Conclusion:</b> Observe the last row 'Outcome' and note its correlation scores with different features. We can observe that Glucose, BMI and Age are the most correlated with Outcome. BloodPressure, Insulin, DiabetesPedigreeFunction are the least correlated so we can drop them.

### Drop least correlated features

In [ ]:
# Drop leaset correlationed variables - BloodPressure, Insulin, DiabetesPedigreeFunction, and SkinThickness
df_selected=df.drop(['BloodPressure','Insulin','DiabetesPedigreeFunction', 'SkinThickness'],axis='columns')


In [ ]:
# Create correlation matrix
df_selected.corr()

## Outliers

Outliers are the extreme value within the dataset and can have a big impact on statistical analysis as the can impact calculations like mean and standard deviation. 

First we'll view a quick numerical breakdown followed by histograms and boxpolots.

In [ ]:
cont_vars = list(df_selected.columns) #[1:]
outliers_visual(df)

Outliers are unusual values in a dataset and they can distort statistical analyses and violate their assumptions.  Hence, it is important to deal with them. Removing outliers can cause data loss so we have to deal with it using various scaling or transformation techniques.

Each feature's box represents it's 50% of the data's interquartile range.  In this case we can ignore the binary target variable (Outcome) boxplot for the most part, there are better ways to describe this target such as the Binary Classification Imbalance section above.  

<b>Observations:</b>
<li>For pregnancies, the 1st 25% quartile has a very low as is the inter quartile range and median.  This of course is because more women will have 0-1 pregancies than 5-10, for instance.  The outliers are in the 12-15 range. No significant skews observed. Because 12-15 pregnancies is a reality there is not reason to remove these outliers.<\li>
<li>Glucose displays a normal distribution with the inter quartile and with no significant skew or outliers.<\li>
<li>SkinThinkess show us a tight inter quartile but with significant outliers.<\li>        
<li>The BMI feature significant outliers past the maximum quartile with a normal distribution with the inter quartile. The outliers are most like not bad data.<\li>    
<li>Age is much like BMI with the outliers being logincally emplainable.<\li>        


In [ ]:
# Outlier_summary(),  count outliers and plot
with open('charts/Outlier_Summary.txt', mode='w', newline='') as outlier_file:
    outlier_writer = csv.writer(outlier_file, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
    for col in df_selected.columns: 
        q75, q25 = np.percentile(df_selected[col], [75, 25])
        iqr = q75 - q25
        min_val = q25 - (iqr * 1.5)
        max_val = q75 + (iqr * 1.5)
        count = len(np.where((df_selected[col] > max_val) | (df[col] < min_val))[0])
        percent = round(count / len(df_selected[col]) * 100, 2)

        total_chars = 45
        len_first_chars = 7
        len_col = len(col)
        chars_needed = total_chars - len_col - len_first_chars

        print(len_first_chars * '-' + col + ' Outliers' + chars_needed * '-')
        print('Count: {}'.format(count))
        print('Percentage of Data: {}%'.format(percent))

        outlier_writer.writerow([len_first_chars * '-' + col + ' Outliers' + chars_needed * '-'])
        outlier_writer.writerow(['Count: {}'.format(count)])
        outlier_writer.writerow(['Percentage of Data: {}%'.format(percent)])
        outlier_writer.writerow(' ')

Next we'll use a sctter matrix to visualize the relationship between the multiple variables in the dataset.

In [ ]:
scatter_matrix(df_selected,figsize=(15,15));

<b>Observations:</b>
<li>If we observe pregnancies with insulin, we can conclude that women with higher number of pregnancies have lower insulin.</li>
<li>There are more pregnancies the younger a women is which is logical.</li>
<li>There is a much lower rate of DiabetesPedegreeFunction with a lower age.  This could indicate that people live a shorter life with DiabetesPedegreeFunction.</li>
<li>Much like DiabetesPedegreeFunction, Glucose is higher amoung yourger people.  Possible because shorter lives or living healthier as people age.</li>

# Train, Test, and Evaluate Model 
So here we are, ready to model and predict!  In this section we'll walk through logistical regression, linear regression, multiple regression, polynomial regression, decision tree, random forest.

## Create Training Data

Let's split the dataset by using the function train_test_split(). You need to pass three parameters features; target, and test_set size.

In [ ]:
target_name='Outcome'
y= df_selected[target_name] 
X=df_selected.drop(target_name,axis=1)

In [ ]:
X.head() # Contains only independent features 

In [ ]:
# Contains dependent feature
y.head() 

Next let's split the dataset into training set and test set, 80% training, 20% testing.

In [ ]:
X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.2,random_state=0)

In [ ]:
X_train.shape,y_train.shape

In [ ]:
X_test.shape,y_test.shape

The dataset containing 768 records has been split in 624 training records and 154 test records

## Predictive and Classification Algorithms

<li>Logistic Regression</li>
<li>KNN</li>
<li>Naive Bayes</li>
<li>SVM</li>
<li>Decision Tree</li>
<li>AdaBoostClassifier</li>
<li>Gradient Boosting Classifier</li>

### Logistic Regression
Because we're dealing with a binary dependant variable (Outcome), we'll use logisctic regression rather than linear.

In [ ]:
reg = LogisticRegression()
reg.fit(X_train,y_train)   

In [ ]:
lr_pred=reg.predict(X_test)

In [ ]:
y_pred=reg.predict(X_test)
print("Classification Report is:\n",classification_report(y_test,y_pred))
print("Training Score: ",round(float(reg.score(X_train,y_train)*100), 4))
print("Mean Squared Error: ",round(float(mean_squared_error(y_test,y_pred)), 4))
print("R2 score is: ",round(float(r2_score(y_test,y_pred)), 4))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))
sns.heatmap(confusion_matrix(y_test,lr_pred))

<b>Observations for Logictic Regression:</b>

<li>Precision: The precision scores which represents the classifiers ability to predict correct predictions relative to total positive predictions. Out of all the people in the dataset the Logictic Regression model says that 77% had diabetes.</li>
<li>Recall: The recall scores which represents correct positive predictions relative to total actual positives. The model only predicted this outcome correctly for 57% of those people.</li>
<li>F1: The F1 score represent a weighted mean of precision and recall. In this case the F1 score is only .63, a far cry from a perfect 1 score.</li>      

### K-Nearest Neighbors
KNN is used next as a non-parametric algorithm that will classify the data points based on their proximity and association to other available data.  KNN is a type of instance-based learning where the function is only approximated locally and all computation is deferred until function evaluation. KNN relies on distance for classification and assumes that similar things are near to each other that nonsimilar things.

In [ ]:
# List Hyperparameters to tune
knn= KNeighborsClassifier()
n_neighbors = list(range(15,25))
p=[1,2]
weights = ['uniform', 'distance']
metric = ['euclidean', 'manhattan', 'minkowski']

# Convert to dictionary
hyperparameters = dict(n_neighbors=n_neighbors, p=p,weights=weights,metric=metric)

# Making model
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
grid_search = GridSearchCV(estimator=knn, param_grid=hyperparameters, n_jobs=-1, cv=cv, scoring='f1',error_score=0)

In [ ]:
best_model = grid_search.fit(X_train,y_train)

The most important KNN hyperparameter is the number of neighbors.

In [ ]:
# Best Hyperparameters Value
print('Best leaf_size:', best_model.best_estimator_.get_params()['leaf_size'])
print('Best p:', best_model.best_estimator_.get_params()['p'])
print('Best n_neighbors:', best_model.best_estimator_.get_params()['n_neighbors'])

In [ ]:
# Predict testing set
knn_pred = best_model.predict(X_test)

In [ ]:
print("Classification Report is:\n",classification_report(y_test,knn_pred))
print("Training Score: ",round(float(reg.score(X_train,y_train)*100), 4))
print("Mean Squared Error: ",round(float(mean_squared_error(y_test,y_pred)), 4))
print("R2 score is: ",round(float(r2_score(y_test,y_pred)), 4))
print("\n Confusion Matrix:\n")
sns.heatmap(confusion_matrix(y_test,knn_pred))

<b>Observations for K-Nearest Neighbors Model:</b>

<li>This means that out of all the people in the dataset the model says that 70% had diabetes</li>
<li>Out of all the people that actually did have diabet, the model only predicted this outcome correctly for 36% of those people.</li>
<li>The closer to 1 the better the score. In this case the F1 score is .67</li>


### Naive Bayes

Naive Bayes is classification approach that adopts the principle of class conditional independence from the Bayes Theorem. This means that the presence of one feature does not impact the presence of another in the probability of a given outcome, and each predictor has an equal effect on that result

In [ ]:
param_grid_nb = {
    'var_smoothing': np.logspace(0,-2, num=100)
}
nbModel_grid = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid_nb, verbose=1, cv=10, n_jobs=-1)

In [ ]:
best_model= nbModel_grid.fit(X_train, y_train)

In [ ]:
nb_pred=best_model.predict(X_test)

In [ ]:
print("Classification Report is:\n",classification_report(y_test,nb_pred))
print("Training Score: ",round(float(reg.score(X_train,y_train)*100), 4))
print("Mean Squared Error: ",round(float(mean_squared_error(y_test,y_pred)), 4))
print("R2 score is: ",round(float(r2_score(y_test,y_pred)), 4))
print("\n Confusion Matrix:\n")
sns.heatmap(confusion_matrix(y_test,nb_pred))

<b>Observations for Naive Bayes Model:</b>

<li>Precision: Out of all the people in the dataset the model says that 64% had diabetes.</li>
<li>Recall: The model only predicted this outcome correctly for 54% of those people.</li>
<li>In this case the F1 score is only .58, a far cry from a perfect 1 score.</li>


### Support Vector Machine
It is typically leveraged for classification problems, constructing a hyperplane where the distance between two classes of data points is at its maximum. This hyperplane is known as the decision boundary, separating the classes of data points (e.g., has diabetes vs doesn't have diabetes ) on either side of the plane.

In [ ]:
model = SVC()
kernel = ['poly', 'rbf', 'sigmoid']
C = [50, 10, 1.0, 0.1, 0.01]
gamma = ['scale']

In [ ]:
# define grid search
grid = dict(kernel=kernel,C=C,gamma=gamma)
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
grid_search = GridSearchCV(estimator=model, param_grid=grid, n_jobs=-1, cv=cv, scoring='f1',error_score=0)

In [ ]:
# define grid search
grid = dict(kernel=kernel,C=C,gamma=gamma)
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
grid_search = GridSearchCV(estimator=model, param_grid=grid, n_jobs=-1, cv=cv, scoring='f1',error_score=0)

In [ ]:
grid_result = grid_search.fit(X, y)

In [ ]:
svm_pred=grid_result.predict(X_test)

In [ ]:
print("Classification Report is:\n",classification_report(y_test,svm_pred))
print("Training Score: ",round(float(reg.score(X_train,y_train)*100), 4))
print("Mean Squared Error: ",round(float(mean_squared_error(y_test,y_pred)), 4))
print("R2 score is: ",round(float(r2_score(y_test,y_pred)), 4))
print("\n Confusion Matrix:\n")
sns.heatmap(confusion_matrix(y_test,svm_pred))

<b>Observations for Support Vector Machine:</b>

<li>Precision: Out of all the people in the dataset the SVM model says that 73% had diabetes.</li>
<li>Recall: The model only predicted this outcome correctly for 68% of those people.</li>
<li>In this case the F1 score is only .70, a far cry from a perfect 1 score.</li>

print("Classification Report is:\n",classification_report(y_test,svm_pred))
print("\n F1:\n",f1_score(y_test,knn_pred))
print("\n Precision score is:\n",precision_score(y_test,knn_pred))
print("\n Recall score is:\n",recall_score(y_test,knn_pred))
print("\n Confusion Matrix:\n")
sns.heatmap(confusion_matrix(y_test,svm_pred))

### Decision Tree

In [ ]:
# Create the parameter grid based on the results of random search 
params = {
    'max_depth': [5, 10, 20,25],
    'min_samples_leaf': [10, 20, 50, 100,120],
    'criterion': ["gini", "entropy"]
}

In [ ]:
grid_search = GridSearchCV(estimator=dt, 
                           param_grid=params, 
                           cv=4, n_jobs=-1, verbose=1, scoring = "accuracy")

In [ ]:
best_model=grid_search.fit(X_train, y_train)

In [ ]:
dt_pred=best_model.predict(X_test)

In [ ]:
print("Classification Report is:\n",classification_report(y_test,dt_pred))
print("Training Score: ",round(float(reg.score(X_train,y_train)*100), 4))
print("Mean Squared Error: ",round(float(mean_squared_error(y_test,y_pred)), 4))
print("R2 score is: ",round(float(r2_score(y_test,y_pred)), 4))
print("\n Confusion Matrix:\n")
sns.heatmap(confusion_matrix(y_test,dt_pred))

<b>Observations for Decision Tree Classification Report:</b>

<li>Precision: Out of all the people in the dataset the Decision Tree model says that 78% had diabetes.</li>
<li>Recall: The model only predicted this outcome correctly for 45% of those people.</li>
<li>In this case the F1 score is only .57, a far cry from a perfect 1 score.</li>     

In [ ]:
# Feature variables
X=df_selected.drop(['Outcome'], axis=1)
X

### Visualizing Decision Tree

In [ ]:
#df_dt=.drop(['BloodPressure','Insulin','DiabetesPedigreeFunction'],axis='columns')
df_selected.head()

In [ ]:
# split dataset in features and target variable
feature_cols = ['Pregnancies', 'Glucose', 'BMI', 'Age']
X = df_selected[feature_cols] # Features
#X = df_selected[coloumns] # Features
y = df_selected.Outcome # Target variable


In [ ]:
# Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1) # 70% training and 30% test

In [ ]:
# Create Decision Tree classifer object
clf = DecisionTreeClassifier()

# Train Decision Tree Classifer
clf = clf.fit(X_train,y_train)

#Predict the response for test dataset
y_pred = clf.predict(X_test)

In [ ]:
dot_data = StringIO()
export_graphviz(clf, out_file=dot_data,  
                filled=True, rounded=True,
                special_characters=True,feature_names = feature_cols,class_names=['0','1'])
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())  
graph.write_png('diabetes.png')
Image(graph.create_png())

### AdaBoostClassifier

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
adb = AdaBoostClassifier(estimator = None)
adb.fit(X_train,y_train)

In [ ]:
y_pred=adb.predict(X_test)
print("Classification Report is:\n",classification_report(y_test,y_pred))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))
print("Training Score:\n",adb.score(X_train,y_train)*100)
print("Mean Squared Error:\n",mean_squared_error(y_test,y_pred))
print("R2 score is:\n",r2_score(y_test,y_pred))

In [ ]:
print(accuracy_score(y_test,y_pred)*100)

<b>Conclusion:</b>



### Gradient Boosting Classifier

In [ ]:
gbc=GradientBoostingClassifier()
gbc.fit(X_train,y_train)

In [ ]:
y_pred=gbc.predict(X_test)
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
print("Classification Report is:\n",classification_report(y_test,y_pred))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))
print("Training Score:\n",gbc.score(X_train,y_train)*100)
print("Mean Squared Error:\n",mean_squared_error(y_test,y_pred))
print("R2 score is:\n",r2_score(y_test,y_pred))

# Conclusion

So now we conclude the accuracy of the different models:
<li>Logistic Regression: 81%</li>
<li>K-Nearest Neighbors Classifier: 80%</li>
<li>Naive Bayes: 79%</li>
<li>Support Vector Machine: 81%</li>
<li>Decision Tree Classifier: 79%</li>
<li>AdaBoostClassifier: 79%<li>
<li>Gradient Boosting Classifier: 79%</li>

Based on the close accuracy scores from each model we have cleaned up our data quite well. Each model ranges between 79% and 81%.  Nice!  Support Vector Machine scored the highest accuracy at 81%.